# Chapter 23: Log Collection and Processing

**From: AI for Networking Engineers - Volume 2**

## Overview

Network devices generate millions of logs daily. This chapter teaches you to:

1. **Collect** logs from multi-vendor devices (Cisco, Juniper, Arista, Palo Alto)
2. **Parse** different syslog formats and extract structured data
3. **Normalize** timestamps across time zones and formats
4. **Deduplicate** logs in high-availability setups
5. **Process** at scale with performance benchmarks

**Real-World Impact**: Proper log processing reduces MTTR by 67% and catches 94% of incidents before user impact.

Let's build a production-ready log processing pipeline!

## Setup: Install Required Libraries

In [ ]:
!pip install -q python-dateutil pytz faker

## Import Required Libraries

In [ ]:
import re
import json
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
from collections import defaultdict, Counter
from dateutil import parser as dateparser
import pytz
import hashlib
import time

## Section 1: Realistic Network Log Dataset

Real network logs from multiple vendors showing common events:

In [ ]:
# Realistic multi-vendor log samples
NETWORK_LOGS = [
    # Cisco IOS Router Logs
    "<189>Mar 15 10:23:45.123 RT-CORE-01 %LINEPROTO-5-UPDOWN: Line protocol on Interface GigabitEthernet0/1, changed state to up",
    "<189>Mar 15 10:24:12.456 RT-CORE-01 %BGP-5-ADJCHANGE: neighbor 10.1.1.2 Up",
    "<187>Mar 15 10:25:33.789 RT-CORE-02 %SYS-5-CONFIG_I: Configured from console by admin on vty0 (10.100.1.50)",
    "<190>Mar 15 10:26:01.234 SW-ACCESS-03 %LINK-3-UPDOWN: Interface GigabitEthernet1/0/24, changed state to down",
    "<186>Mar 15 10:26:15.567 SW-ACCESS-03 %LINK-3-UPDOWN: Interface GigabitEthernet1/0/24, changed state to up",
    "<183>Mar 15 10:27:45.890 FW-EDGE-01 %SEC-6-IPACCESSLOGP: list 101 denied tcp 192.168.1.100(55123) -> 10.0.0.5(22), 1 packet",
    "<189>Mar 15 10:28:30.123 RT-CORE-01 %OSPF-5-ADJCHG: Process 1, Nbr 10.2.2.2 on GigabitEthernet0/2 from LOADING to FULL",
    "<184>Mar 15 10:29:15.456 SW-CORE-01 %SPANTREE-2-ROOTGUARD_BLOCK: Root guard blocking port GigabitEthernet1/1 on VLAN100",
    "<189>Mar 15 10:30:00.789 RT-WAN-01 %DUAL-5-NBRCHANGE: EIGRP-IPv4 100: Neighbor 10.3.3.3 (GigabitEthernet0/1) is up",
    "<187>Mar 15 10:31:22.012 SW-DIST-02 %STORM_CONTROL-3-SHUTDOWN: A packet storm was detected on Gi2/0/10. The interface has been disabled",
    
    # Juniper Junos Logs
    "<28>Mar 15 10:32:10 RT-EDGE-JNX01 rpd[1234]: RPD_OSPF_NBRDOWN: OSPF neighbor 10.4.4.4 (ge-0/0/1.0) state changed from Full to Down",
    "<30>Mar 15 10:33:05 SW-AGG-JNX02 chassisd[5678]: CHASSISD_SNMP_TRAP: FRU power supply 0 OK",
    "<29>Mar 15 10:34:18 FW-CORE-JNX01 mib2d[9012]: SNMP_TRAP_LINK_DOWN: ifIndex 512, ifAdminStatus up(1), ifOperStatus down(2)",
    "<28>Mar 15 10:35:42 RT-EDGE-JNX01 rpd[1234]: BGP_PREFIX_THRESH_EXCEEDED: 10.5.5.5 (External AS 65002): Configured maximum prefix-limit threshold(90) exceeded",
    "<30>Mar 15 10:36:55 SW-ACCESS-JNX03 eventd[3456]: EVENTD_AUDIT_COMMIT: User 'netops' commit configuration",
    
    # Arista EOS Logs (JSON format)
    '{"timestamp": "2024-03-15T10:37:30.123Z", "hostname": "SW-SPINE-ARI01", "severity": "warning", "facility": "LINEPROTO", "message": "Line protocol on Interface Ethernet1/1 changed state to up"}',
    '{"timestamp": "2024-03-15T10:38:15.456Z", "hostname": "SW-LEAF-ARI02", "severity": "info", "facility": "BGP", "message": "BGP peer 10.6.6.6 state changed to Established"}',
    '{"timestamp": "2024-03-15T10:39:00.789Z", "hostname": "SW-SPINE-ARI01", "severity": "error", "facility": "MLAG", "message": "MLAG peer-link interface down"}',
    
    # Palo Alto Firewall Logs
    "Mar 15 10:40:12 PA-FW-01 1,2024/03/15 10:40:12,009401000001,TRAFFIC,end,2304,2024/03/15 10:40:12,192.168.10.50,8.8.8.8,0.0.0.0,0.0.0.0,Allow_Internet,,,dns,vsys1,trust,untrust,ethernet1/1,ethernet1/2,Log Forwarding,2024/03/15 10:40:12,45123,1,53,53,0,0,0x19,udp,allow,78,78,0,1,2024/03/15 10:40:12,0,any,0,1234567890,0x0,10.0.0.0-10.255.255.255,8.0.0.0-8.255.255.255,0,1,0,policy-deny,0,0,0,0,,PA-FW-01,from-policy,,,0,,0,,N/A,0,0,0,0",
    "Mar 15 10:41:33 PA-FW-01 1,2024/03/15 10:41:33,009401000001,THREAT,vulnerability,2304,2024/03/15 10:41:33,172.16.50.100,203.0.113.45,0.0.0.0,0.0.0.0,Inbound_Rules,,,web-browsing,vsys1,dmz,untrust,ethernet1/3,ethernet1/2,Alert Profile,2024/03/15 10:41:33,67890,1,443,443,0,0,0x2000,tcp,alert,1024,512,512,1,2024/03/15 10:41:33,0,any,0,9876543210,0x8000000000000000,Germany,United States,0,,HTTP SQL Injection(12345),informational,client-to-server,8765432109,0x0,10.0.0.0-10.255.255.255,200.0.0.0-200.255.255.255",
    
    # More Cisco - Interface flapping scenario
    "<190>Mar 15 10:42:01.111 SW-ACCESS-05 %LINK-3-UPDOWN: Interface GigabitEthernet1/0/12, changed state to down",
    "<186>Mar 15 10:42:02.222 SW-ACCESS-05 %LINK-3-UPDOWN: Interface GigabitEthernet1/0/12, changed state to up",
    "<190>Mar 15 10:42:05.333 SW-ACCESS-05 %LINK-3-UPDOWN: Interface GigabitEthernet1/0/12, changed state to down",
    "<186>Mar 15 10:42:06.444 SW-ACCESS-05 %LINK-3-UPDOWN: Interface GigabitEthernet1/0/12, changed state to up",
    "<190>Mar 15 10:42:09.555 SW-ACCESS-05 %LINK-3-UPDOWN: Interface GigabitEthernet1/0/12, changed state to down",
    
    # Authentication and Security Events
    "<182>Mar 15 10:43:20.666 SW-ACCESS-07 %AUTHMGR-5-SUCCESS: Authorization succeeded for client (0050.5699.1234) on Interface Gi1/0/5 AuditSessionID 0A64010200000001",
    "<181>Mar 15 10:44:15.777 SW-ACCESS-07 %AUTHMGR-7-FAILOVER: Failing over from method dot1x to method mab for client (0050.5699.5678) on Interface Gi1/0/8",
    "<180>Mar 15 10:45:30.888 FW-EDGE-02 %SEC-4-IPACCESSLOGDP: list 150 denied icmp 10.200.50.25 -> 10.0.1.1 (type 8, code 0), 5 packets",
    
    # BGP Events
    "<189>Mar 15 10:46:45.999 RT-CORE-03 %BGP-5-ADJCHANGE: neighbor 10.7.7.7 Down BGP Notification sent",
    "<189>Mar 15 10:47:12.000 RT-CORE-03 %BGP-5-ADJCHANGE: neighbor 10.7.7.7 Up",
    "<188>Mar 15 10:48:30.111 RT-WAN-02 %BGP-4-MAXPFX: No. of prefix received from 10.8.8.8 (afi 0): 950 exceed limit 1000"
]

print(f"✓ Loaded {len(NETWORK_LOGS)} realistic network logs from 4 vendors")
print(f"  - Cisco IOS: Routers, switches, firewalls")
print(f"  - Juniper Junos: Routing and switching")
print(f"  - Arista EOS: Data center fabric")
print(f"  - Palo Alto: Next-gen firewall")

## Section 2: Multi-Vendor Log Parser

Detect vendor and parse logs into structured format:

In [ ]:
class MultiVendorLogParser:
    """
    Auto-detect vendor and parse network logs
    """
    
    # Cisco syslog pattern: <PRI>TIMESTAMP HOSTNAME %FACILITY-SEVERITY-MNEMONIC: MESSAGE
    CISCO_PATTERN = re.compile(
        r'<(?P<priority>\d+)>(?P<timestamp>\w+ \d+ \d+:\d+:\d+\.\d+) '
        r'(?P<hostname>\S+) %(?P<facility>\w+)-(?P<severity>\d+)-(?P<mnemonic>\w+): '
        r'(?P<message>.+)'
    )
    
    # Juniper syslog pattern: <PRI>TIMESTAMP HOSTNAME PROCESS[PID]: MNEMONIC: MESSAGE
    JUNIPER_PATTERN = re.compile(
        r'<(?P<priority>\d+)>(?P<timestamp>\w+ \d+ \d+:\d+:\d+) '
        r'(?P<hostname>\S+) (?P<process>\w+)\[(?P<pid>\d+)\]: '
        r'(?P<mnemonic>[\w_]+): (?P<message>.+)'
    )
    
    def __init__(self):
        self.stats = Counter()
    
    def detect_vendor(self, log_line: str) -> str:
        """Auto-detect vendor from log format"""
        
        # Check for JSON (Arista)
        if log_line.strip().startswith('{'):
            return 'arista'
        
        # Check for Palo Alto (starts with date or has many commas)
        if log_line.count(',') > 20:
            return 'palo_alto'
        
        # Check for Juniper (process[pid] pattern)
        if re.search(r'\w+\[\d+\]:', log_line):
            return 'juniper'
        
        # Default to Cisco (most common)
        return 'cisco'
    
    def parse_cisco(self, log_line: str) -> Optional[Dict]:
        """Parse Cisco IOS syslog"""
        match = self.CISCO_PATTERN.match(log_line)
        if not match:
            return None
        
        return {
            'vendor': 'cisco',
            'priority': int(match.group('priority')),
            'timestamp': match.group('timestamp'),
            'hostname': match.group('hostname'),
            'facility': match.group('facility'),
            'severity': int(match.group('severity')),
            'mnemonic': match.group('mnemonic'),
            'message': match.group('message'),
            'raw': log_line
        }
    
    def parse_juniper(self, log_line: str) -> Optional[Dict]:
        """Parse Juniper Junos syslog"""
        match = self.JUNIPER_PATTERN.match(log_line)
        if not match:
            return None
        
        return {
            'vendor': 'juniper',
            'priority': int(match.group('priority')),
            'timestamp': match.group('timestamp'),
            'hostname': match.group('hostname'),
            'process': match.group('process'),
            'pid': int(match.group('pid')),
            'mnemonic': match.group('mnemonic'),
            'message': match.group('message'),
            'raw': log_line
        }
    
    def parse_arista(self, log_line: str) -> Optional[Dict]:
        """Parse Arista EOS JSON logs"""
        try:
            data = json.loads(log_line)
            return {
                'vendor': 'arista',
                'timestamp': data.get('timestamp'),
                'hostname': data.get('hostname'),
                'severity': data.get('severity'),
                'facility': data.get('facility'),
                'message': data.get('message'),
                'raw': log_line
            }
        except json.JSONDecodeError:
            return None
    
    def parse_palo_alto(self, log_line: str) -> Optional[Dict]:
        """Parse Palo Alto firewall logs (simplified)"""
        parts = log_line.split(',')
        if len(parts) < 10:
            return None
        
        return {
            'vendor': 'palo_alto',
            'timestamp': parts[1] if len(parts) > 1 else '',
            'serial': parts[2] if len(parts) > 2 else '',
            'log_type': parts[3] if len(parts) > 3 else '',
            'subtype': parts[4] if len(parts) > 4 else '',
            'src_ip': parts[7] if len(parts) > 7 else '',
            'dst_ip': parts[8] if len(parts) > 8 else '',
            'action': 'allow' if 'allow' in log_line.lower() else 'deny',
            'raw': log_line
        }
    
    def parse(self, log_line: str) -> Optional[Dict]:
        """Parse log based on detected vendor"""
        vendor = self.detect_vendor(log_line)
        self.stats[vendor] += 1
        
        parsers = {
            'cisco': self.parse_cisco,
            'juniper': self.parse_juniper,
            'arista': self.parse_arista,
            'palo_alto': self.parse_palo_alto
        }
        
        parser = parsers.get(vendor)
        if parser:
            result = parser(log_line)
            if result:
                self.stats['parsed_success'] += 1
                return result
        
        self.stats['parse_failed'] += 1
        return None

# Test the parser
parser = MultiVendorLogParser()

print("\n" + "="*60)
print("Testing Multi-Vendor Parser")
print("="*60)

# Parse all logs
parsed_logs = []
for log in NETWORK_LOGS:
    result = parser.parse(log)
    if result:
        parsed_logs.append(result)

# Show statistics
print(f"\nParsing Statistics:")
print(f"  Total logs: {len(NETWORK_LOGS)}")
print(f"  Successfully parsed: {parser.stats['parsed_success']}")
print(f"  Failed to parse: {parser.stats['parse_failed']}")
print(f"\nVendor breakdown:")
for vendor in ['cisco', 'juniper', 'arista', 'palo_alto']:
    count = parser.stats[vendor]
    print(f"  {vendor.title()}: {count}")

# Show sample parsed log
print(f"\nSample Parsed Log (Cisco):")
cisco_log = next((log for log in parsed_logs if log['vendor'] == 'cisco'), None)
if cisco_log:
    for key, value in cisco_log.items():
        if key != 'raw':
            print(f"  {key}: {value}")

## Section 3: Timestamp Normalization

Handle multiple timestamp formats and normalize to UTC:

In [ ]:
class TimestampNormalizer:
    """
    Normalize timestamps from different formats to UTC datetime
    """
    
    def __init__(self, default_year=None, default_tz='UTC'):
        self.default_year = default_year or datetime.now().year
        self.default_tz = pytz.timezone(default_tz)
    
    def normalize(self, timestamp_str: str, vendor: str = 'cisco') -> datetime:
        """
        Normalize timestamp to UTC datetime object
        """
        
        # Arista uses ISO 8601 format
        if vendor == 'arista' and 'T' in timestamp_str:
            return dateparser.isoparse(timestamp_str)
        
        # Cisco/Juniper: "Mar 15 10:23:45.123" (no year)
        if vendor in ['cisco', 'juniper']:
            # Parse with assumed year
            try:
                dt = datetime.strptime(f"{self.default_year} {timestamp_str}", 
                                        "%Y %b %d %H:%M:%S.%f")
                # Localize to default timezone, then convert to UTC
                dt_localized = self.default_tz.localize(dt)
                return dt_localized.astimezone(pytz.UTC)
            except ValueError:
                # Fallback: without milliseconds
                dt = datetime.strptime(f"{self.default_year} {timestamp_str}", 
                                        "%Y %b %d %H:%M:%S")
                dt_localized = self.default_tz.localize(dt)
                return dt_localized.astimezone(pytz.UTC)
        
        # Palo Alto: "2024/03/15 10:40:12"
        if vendor == 'palo_alto':
            dt = datetime.strptime(timestamp_str, "%Y/%m/%d %H:%M:%S")
            dt_localized = self.default_tz.localize(dt)
            return dt_localized.astimezone(pytz.UTC)
        
        # Fallback: Try automatic parsing
        return dateparser.parse(timestamp_str)

# Test timestamp normalization
normalizer = TimestampNormalizer(default_year=2024, default_tz='US/Eastern')

print("\n" + "="*60)
print("Testing Timestamp Normalization")
print("="*60)

test_timestamps = [
    ('Mar 15 10:23:45.123', 'cisco'),
    ('2024-03-15T10:37:30.123Z', 'arista'),
    ('2024/03/15 10:40:12', 'palo_alto'),
]

for ts_str, vendor in test_timestamps:
    normalized = normalizer.normalize(ts_str, vendor)
    print(f"\n{vendor.title()}: {ts_str}")
    print(f"  → Normalized (UTC): {normalized}")
    print(f"  → ISO format: {normalized.isoformat()}")

## Section 4: Log Deduplication

Critical for high-availability setups with redundant collectors:

In [ ]:
class LogDeduplicator:
    """
    Eliminate duplicate logs in redundant collector setups
    """
    
    def __init__(self, window_seconds=300):
        self.window_seconds = window_seconds
        self.seen_hashes = {}  # hash -> first_seen_time
        self.stats = Counter()
    
    def _hash_log(self, log_data: Dict) -> str:
        """Create hash from log content (excluding timestamp)"""
        # Use hostname + message for uniqueness
        key_parts = [
            log_data.get('hostname', ''),
            log_data.get('message', ''),
            log_data.get('mnemonic', '')
        ]
        key = '|'.join(key_parts)
        return hashlib.md5(key.encode()).hexdigest()
    
    def is_duplicate(self, log_data: Dict, current_time: datetime) -> bool:
        """
        Check if log is duplicate within time window
        """
        log_hash = self._hash_log(log_data)
        
        # Clean old entries
        cutoff_time = current_time - timedelta(seconds=self.window_seconds)
        self.seen_hashes = {
            h: t for h, t in self.seen_hashes.items() 
            if t > cutoff_time
        }
        
        # Check if we've seen this log recently
        if log_hash in self.seen_hashes:
            self.stats['duplicates'] += 1
            return True
        
        # Mark as seen
        self.seen_hashes[log_hash] = current_time
        self.stats['unique'] += 1
        return False

# Test deduplication with simulated duplicates
deduplicator = LogDeduplicator(window_seconds=60)

print("\n" + "="*60)
print("Testing Log Deduplication")
print("="*60)

# Simulate redundant collectors sending same logs
test_logs = parsed_logs[:5]  # First 5 logs
duplicate_logs = parsed_logs[:5]  # Same 5 logs again (duplicates)

current_time = datetime.now(pytz.UTC)

print(f"\nProcessing {len(test_logs)} original logs...")
for log in test_logs:
    is_dup = deduplicator.is_duplicate(log, current_time)

print(f"Processing {len(duplicate_logs)} duplicate logs...")
for log in duplicate_logs:
    current_time += timedelta(seconds=1)
    is_dup = deduplicator.is_duplicate(log, current_time)

print(f"\nDeduplication Results:")
print(f"  Unique logs: {deduplicator.stats['unique']}")
print(f"  Duplicates blocked: {deduplicator.stats['duplicates']}")
print(f"  Dedup rate: {deduplicator.stats['duplicates'] / (deduplicator.stats['unique'] + deduplicator.stats['duplicates']) * 100:.1f}%")

## Section 5: Complete Log Processing Pipeline

Combine all components into a production-ready pipeline:

In [ ]:
class LogProcessingPipeline:
    """
    Production log processing pipeline
    """
    
    def __init__(self):
        self.parser = MultiVendorLogParser()
        self.normalizer = TimestampNormalizer(default_year=2024)
        self.deduplicator = LogDeduplicator(window_seconds=300)
        self.processed_logs = []
        self.stats = Counter()
    
    def process_log(self, raw_log: str) -> Optional[Dict]:
        """
        Process a single log through the pipeline
        """
        self.stats['total_received'] += 1
        
        # Step 1: Parse
        parsed = self.parser.parse(raw_log)
        if not parsed:
            self.stats['parse_failed'] += 1
            return None
        
        # Step 2: Normalize timestamp
        try:
            normalized_ts = self.normalizer.normalize(
                parsed['timestamp'], 
                parsed['vendor']
            )
            parsed['timestamp_utc'] = normalized_ts
        except Exception as e:
            self.stats['timestamp_failed'] += 1
            parsed['timestamp_utc'] = datetime.now(pytz.UTC)
        
        # Step 3: Deduplicate
        if self.deduplicator.is_duplicate(parsed, parsed['timestamp_utc']):
            self.stats['duplicates'] += 1
            return None
        
        # Step 4: Enrich (add processing metadata)
        parsed['processed_at'] = datetime.now(pytz.UTC).isoformat()
        parsed['pipeline_version'] = '1.0'
        
        self.stats['processed_success'] += 1
        self.processed_logs.append(parsed)
        return parsed
    
    def process_batch(self, raw_logs: List[str]) -> List[Dict]:
        """
        Process multiple logs
        """
        results = []
        for log in raw_logs:
            result = self.process_log(log)
            if result:
                results.append(result)
        return results
    
    def get_statistics(self) -> Dict:
        """
        Return processing statistics
        """
        return {
            'total_received': self.stats['total_received'],
            'processed_success': self.stats['processed_success'],
            'parse_failed': self.stats['parse_failed'],
            'duplicates': self.stats['duplicates'],
            'timestamp_failed': self.stats['timestamp_failed'],
            'success_rate': self.stats['processed_success'] / max(self.stats['total_received'], 1) * 100
        }

# Run the complete pipeline
pipeline = LogProcessingPipeline()

print("\n" + "="*60)
print("Running Complete Log Processing Pipeline")
print("="*60)

# Process all logs
start_time = time.time()
processed = pipeline.process_batch(NETWORK_LOGS)
processing_time = time.time() - start_time

# Display statistics
stats = pipeline.get_statistics()
print(f"\nProcessing Statistics:")
for key, value in stats.items():
    if 'rate' in key:
        print(f"  {key.replace('_', ' ').title()}: {value:.1f}%")
    else:
        print(f"  {key.replace('_', ' ').title()}: {value}")

print(f"\nPerformance:")
print(f"  Total processing time: {processing_time*1000:.2f}ms")
print(f"  Logs per second: {len(NETWORK_LOGS)/processing_time:.0f}")
print(f"  Average time per log: {processing_time/len(NETWORK_LOGS)*1000:.2f}ms")

## Section 6: Log Analysis and Insights

Analyze processed logs to find patterns and issues:

In [ ]:
def analyze_logs(processed_logs: List[Dict]):
    """
    Analyze processed logs for insights
    """
    print("\n" + "="*60)
    print("Log Analysis Report")
    print("="*60)
    
    # 1. Logs by vendor
    vendor_counts = Counter(log['vendor'] for log in processed_logs)
    print(f"\n1. Distribution by Vendor:")
    for vendor, count in vendor_counts.most_common():
        print(f"   {vendor.title()}: {count} logs ({count/len(processed_logs)*100:.1f}%)")
    
    # 2. Most common facilities (Cisco)
    cisco_logs = [log for log in processed_logs if log['vendor'] == 'cisco']
    if cisco_logs:
        facility_counts = Counter(log.get('facility', '') for log in cisco_logs)
        print(f"\n2. Most Common Facilities (Cisco):")
        for facility, count in facility_counts.most_common(5):
            print(f"   {facility}: {count} events")
    
    # 3. Severity distribution (Cisco)
    if cisco_logs:
        severity_counts = Counter(log.get('severity', 0) for log in cisco_logs)
        severity_names = {
            0: 'Emergency', 1: 'Alert', 2: 'Critical', 3: 'Error',
            4: 'Warning', 5: 'Notice', 6: 'Informational', 7: 'Debug'
        }
        print(f"\n3. Severity Distribution (Cisco):")
        for sev, count in sorted(severity_counts.items()):
            print(f"   {sev} ({severity_names.get(sev, 'Unknown')}): {count} events")
    
    # 4. Interface flapping detection
    interface_events = [log for log in cisco_logs if 'Interface' in log.get('message', '')]
    interface_changes = defaultdict(list)
    for log in interface_events:
        msg = log['message']
        # Extract interface name
        import re
        match = re.search(r'Interface (\S+),', msg)
        if match:
            interface = match.group(1)
            interface_changes[interface].append({
                'timestamp': log['timestamp_utc'],
                'state': 'up' if 'up' in msg else 'down',
                'hostname': log['hostname']
            })
    
    print(f"\n4. Interface Flapping Detection:")
    for interface, events in interface_changes.items():
        if len(events) >= 3:
            print(f"   ⚠️  {events[0]['hostname']} {interface}: {len(events)} state changes (FLAPPING!)")
            for event in events[:3]:
                print(f"      - {event['timestamp']}: {event['state']}")
    
    # 5. Security events
    security_keywords = ['denied', 'failed', 'blocked', 'unauthorized']
    security_logs = [
        log for log in processed_logs 
        if any(keyword in log.get('message', '').lower() for keyword in security_keywords)
    ]
    print(f"\n5. Security Events:")
    print(f"   Found {len(security_logs)} security-related events")
    if security_logs:
        for log in security_logs[:3]:
            print(f"   - {log['hostname']}: {log.get('message', '')[:80]}...")

# Run analysis
analyze_logs(pipeline.processed_logs)

## Summary and Key Takeaways

### What We Built:

1. **Multi-Vendor Parser** - Handles Cisco, Juniper, Arista, Palo Alto
2. **Timestamp Normalization** - UTC conversion across formats
3. **Deduplication Engine** - Time-window based duplicate removal
4. **Processing Pipeline** - Production-ready log processing
5. **Log Analysis** - Pattern detection and security insights

### Production Best Practices:

- **Always normalize timestamps** to UTC for correlation
- **Implement deduplication** in HA setups (redundant collectors)
- **Track parsing statistics** to identify format changes
- **Handle vendor-specific formats** with dedicated parsers
- **Batch process** for better performance (1000+ logs/sec)
- **Enrich logs** with context (device inventory, geolocation)
- **Monitor pipeline health** (parse success rate, latency)

### Real-World Applications:

1. **Incident Detection** - Interface flapping, BGP instability
2. **Security Analysis** - Failed auth attempts, denied traffic
3. **Compliance** - Centralized audit trail with retention
4. **Capacity Planning** - Interface utilization trends
5. **Troubleshooting** - Correlate events across devices

### Next Steps:

- Add more vendors (Fortinet, CheckPoint, F5)
- Implement NetFlow/IPFIX parsing
- Build alerting rules engine
- Add ML-based anomaly detection
- Scale to distributed processing (Kafka, Spark)